# Final Project — Grover's Algorithm on Real Quantum Hardware
**PHYS 349 — Individual Assignment**

---

**Due: Thursday, May 8 at midnight**

This is the entire final project — there is no exam. You'll port your Grover's algorithm code from HW 4.6 to a real IBM quantum processor, study how the success probability degrades as circuit depth grows, and quantify *why* by referencing the device's gate error rates and decoherence times.

**The science question:** *At what point does hardware noise overwhelm the quantum advantage?*

---

**How this notebook is organized:**

- **Part A — Bring forward your HW 4.6 code.** Copy your working `build_oracle` and `build_diffusion` functions from HW 4.6 into the cells below and verify them on the simulator. Runs entirely on your laptop; no IBM account needed.
- **Part B — IBM Cloud setup.** One-time account creation and credential save. Detailed walkthrough included.
- **Part C — Run on real hardware.** Submit Grover's at n=2 and n=3 to a real IBM backend; compare success probability to the perfect simulator.
- **Part D — Iteration sweep.** Vary the number of Grover iterations at n=3 to see how the theoretical sin² overshoot curve degrades on real hardware.
- **Part E — Backend properties.** Pull T₁, T₂, and gate errors from the *specific qubits* your circuits used.
- **Part F — Memo to a lab director.** A structured 200–250 word memo recommending whether to run a larger Grover's search on this backend next month.
- **Part G — Experimental design.** Plan a follow-up experiment to pin down this backend's "quantum uselessness threshold."

> ⚠️ **Develop on the simulator first.** Get every circuit working on `AerSimulator` before sending it to real hardware. Queue waits can be minutes to hours, and every job uses your monthly QPU minutes.

**Estimated time:** 4–5 hours of focused work, plus 1–4 hours of queue wait depending on demand. Do not start the night before.

In [ ]:
# === Standard imports ===
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector, Operator
from qiskit.circuit.library import MCMTGate, ZGate
from qiskit_aer import AerSimulator
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.visualization import plot_histogram
import numpy as np
import matplotlib.pyplot as plt
import math

# Local simulator for development and comparison (no IBM account needed)
simulator = AerSimulator()

SHOTS = 4096
print("✓ Imports loaded. Simulator ready.")

---
## Part A — Bring forward your HW 4.6 code

You already wrote `build_oracle(target, n)` and `build_diffusion(n)` in HW 4.6 (Parts A1 and A2). Paste your working versions into the cells below. If you didn't finish HW 4.6, see the lecture solution notebook for reference implementations.

These two functions are the entire algorithmic content of Grover's algorithm — everything else in this notebook is glue, transpilation, and analysis.

### Aside: what is `MCMTGate(ZGate(), n-1, 1)`?

`MCMTGate` (Multi-Controlled Multi-Target) wraps any single-qubit gate with multiple controls.
`MCMTGate(ZGate(), n-1, 1)` means "apply Z to one target, controlled by `n-1` other qubits."
The Z gate applies a −1 phase only when its qubit is in `|1⟩`, so the whole MCMTGate flips the
phase only when **all n qubits are |1⟩**.

For small n it has familiar names:
- n=2 → CZ (controlled-Z)
- n=3 → CCZ (Toffoli with Z instead of X — Hadamards on either side give the standard Toffoli)
- n≥4 → no short name; the chip has to decompose it into many CZs and ancilla swaps

Run the cell below to see what `MCMTGate(ZGate(), 2, 1)` (n=3) looks like under the hood —
this is why the transpiled depth explodes at n=3 and beyond.

In [ ]:
# Peek at the gate decomposition for n=3
demo = QuantumCircuit(3)
demo.append(MCMTGate(ZGate(), 2, 1), range(3))
demo.decompose(reps=2).draw('mpl', fold=-1)

### Worked example: oracle for n=2 marking |11⟩

Before generalizing, let's build the n=2 case explicitly. For target `'11'` on 2 qubits, both
qubits should already be `|1⟩` when we apply the multi-controlled Z, so no X gates are needed:

In [ ]:
# Worked example: oracle for n=2 with target '11' (do not modify)
def build_oracle_n2_for_11():
    qc = QuantumCircuit(2)
    qc.append(MCMTGate(ZGate(), 1, 1), [0, 1])  # = CZ for n=2
    return qc

oracle_demo = build_oracle_n2_for_11()
diag_demo = np.real(np.diag(Operator(oracle_demo).data))
print("Diagonal of n=2 oracle for |11⟩:", diag_demo)
print("Should be [1, 1, 1, -1] — only |11⟩ flipped")
oracle_demo.draw('mpl')

In [ ]:
# === build_oracle  —  generalize the worked example above to arbitrary n ===
# This is the version your full Grover circuit will use. The worked example above
# handles n=2 with target '11'. To generalize:
#   - For each '0' bit in `target`, sandwich the multi-controlled Z with X gates
#     on that qubit so the MCZ fires when the qubit is logically '0'.
#   - Use MCMTGate(ZGate(), n-1, 1) for the controlled-Z over all n qubits.
#
# Recipe reminder:
#   1. Apply X gates to each qubit whose target bit is '0'
#   2. Apply multi-controlled Z (MCMTGate(ZGate(), n-1, 1))
#   3. Apply the same X gates again to undo step 1
#
# Qiskit ordering: target[0] is the MOST significant bit (highest qubit index).
# So target[i] corresponds to qubit (n-1-i).

def build_oracle(target, n):
    """Phase oracle that flips the phase of |target⟩.

    Parameters:
        target (str): bitstring of length n (e.g., '101')
        n (int):     number of qubits
    Returns:
        QuantumCircuit: the phase oracle
    """
    ### YOUR CODE HERE ###
    pass


# Sanity check: oracle for |11⟩ on 2 qubits should give -1 only on |11⟩
oracle_n2 = build_oracle("11", 2)
if oracle_n2 is None:
    raise NotImplementedError("Fill in build_oracle above (replace `pass`) before running this cell.")
diag = np.real(np.diag(Operator(oracle_n2).data))
print("n=2, target='11' diagonal:", diag)
assert np.allclose(diag, [1, 1, 1, -1]), "Oracle should give -1 only on |11⟩"
print("✓ n=2 oracle verified")

# And for |101⟩ on 3 qubits
oracle_n3 = build_oracle("101", 3)
diag3 = np.real(np.diag(Operator(oracle_n3).data))
print("\nn=3, target='101' diagonal:", diag3)
expected3 = np.ones(8); expected3[5] = -1   # |101⟩ has integer index 5
assert np.allclose(diag3, expected3), "Oracle should give -1 only on |101⟩"
print("✓ n=3 oracle verified")

In [ ]:
# === build_diffusion  (from HW 4.6 Part A2) ===
# Paste your working implementation from HW 4.6 below.
#
# Recipe: D = H^⊗n · Z_OR · H^⊗n
# where Z_OR flips phases of every state EXCEPT |0...0⟩.
# Build Z_OR with: X on all → multi-controlled Z → X on all

def build_diffusion(n):
    """The n-qubit diffusion operator: reflection about |s⟩ = H^⊗n |0^n⟩."""
    ### YOUR CODE HERE ###
    pass


# Sanity check: diffusion matrix should equal (2/N) J - I  where J is all-ones
for n in (2, 3):
    N = 2**n
    expected = (2/N) * np.ones((N, N)) - np.eye(N)
    diff_n = build_diffusion(n)
    if diff_n is None:
        raise NotImplementedError("Fill in build_diffusion above (replace `pass`) before running this cell.")
    actual = np.real(Operator(diff_n).data)
    assert np.allclose(actual, expected, atol=1e-9), f"Diffusion mismatch at n={n}"
    print(f"✓ n={n} diffusion verified (matches 2/N · J - I)")

In [ ]:
# === Provided helpers: optimal_iterations + build_grover ===
# These are just glue — you don't need to write them.

def optimal_iterations(n, M=1):
    """Optimal number of Grover iterations for n qubits and M marked states.

    Uses the exact angle formula t = round(π/(4θ) − ½) with θ = asin(√(M/N)).
    The looser round((π/4)·√(N/M)) overshoots for small N (e.g., n=2).
    """
    N = 2**n
    theta = math.asin(math.sqrt(M / N))
    return max(1, round(math.pi / (4 * theta) - 0.5))

def build_grover(target, n, t=None):
    """Full Grover circuit: H^⊗n init, then t iterations of (oracle + diffusion), then measure."""
    if t is None:
        t = optimal_iterations(n)
    qc = QuantumCircuit(n, n)
    qc.h(range(n))                          # equal superposition |s⟩
    oracle    = build_oracle(target, n)
    diffusion = build_diffusion(n)
    for _ in range(t):
        qc.compose(oracle,    inplace=True)
        qc.compose(diffusion, inplace=True)
    qc.measure(range(n), range(n))
    return qc

# Build Grover circuits for n=2 (mark |11⟩) and n=3 (mark |101⟩)
grover_n2 = build_grover("11",   2)
grover_n3 = build_grover("101",  3)
grover_n4 = build_grover("1010", 4)   # n=4: a tougher test for the chip

for n, qc in [(2, grover_n2), (3, grover_n3), (4, grover_n4)]:
    print(f"n={n}:  optimal iterations = {optimal_iterations(n)},   circuit depth = {qc.depth()}")

grover_n3.draw(output='mpl', style='iqp')

In [ ]:
# === Verify on the local simulator (no QPU time used) ===
# If success probability is not >> 1/N for any of the three, fix your build_oracle
# / build_diffusion before continuing.

# AerSimulator can't run MCMTGate directly — transpile to its basis first.
sim_counts = {
    'n=2': simulator.run(transpile(grover_n2, simulator), shots=SHOTS).result().get_counts(),
    'n=3': simulator.run(transpile(grover_n3, simulator), shots=SHOTS).result().get_counts(),
    'n=4': simulator.run(transpile(grover_n4, simulator), shots=SHOTS).result().get_counts(),
}

success_sim = {
    'n=2': sim_counts['n=2'].get('11',   0) / SHOTS,
    'n=3': sim_counts['n=3'].get('101',  0) / SHOTS,
    'n=4': sim_counts['n=4'].get('1010', 0) / SHOTS,
}

print("Simulator success probability (target marked):")
for name, p in success_sim.items():
    print(f"  {name}:  P = {p:.3f}   (expected ≈ 1.0)")

assert success_sim['n=2'] > 0.95, "n=2 success too low — check your oracle"
assert success_sim['n=3'] > 0.85, "n=3 success too low — check your oracle/diffusion"
assert success_sim['n=4'] > 0.80, "n=4 success too low — check your oracle/diffusion"
print("\n✓ All three Grover circuits work on the simulator. Safe to send to hardware.")

---
## Part B — Connect to IBM Quantum Hardware

Everything above runs on your laptop. From here on you'll send circuits to a **real quantum processor** on IBM Cloud.

### One-time setup (do this once per laptop)

**1. Create an IBM Quantum account.**  
Go to [quantum.cloud.ibm.com](https://quantum.cloud.ibm.com) and log in. Google OAuth with your university or personal Gmail is fastest.

**2. Create the free Open Plan instance.**  
On first login, a "Welcome" modal appears. Leave the defaults (`open-instance`, `us-east`, Open Plan), check the license-agreement box, and click **Create instance**. 
If you dismissed the modal: go to the Account Instances section on the home page and click Create an instance +. Either way, select the Open Plan (free, no credit card needed), which gives you 10 QPU minutes per month. Do not select a paid plan.

> If you see a banner saying "Your trial account expires in 29 days" — that refers to the IBM Cloud trial tier, *not* the Open Plan for quantum. The Open Plan is permanently free. **Do not click "Upgrade."**

**3. Create an API key.**  
In the **API key** section on the home page, click **Create +**. Name it something like `phys349-laptop`. **Click Download** — the key is shown only once and disappears after ≈5 minutes. The downloaded file `apikey.json` lands in your `~/Downloads` folder.

**4. Copy your instance CRN.**  
Go to **Instances** → click `open-instance` → click the copy button next to the CRN. The CRN is a long string starting with `crn:v1:bluemix:...` and **ending with `::`** — the double colon is correct, do *not* trim it.

**5. Save credentials by running the next cell.**  
Uncomment, paste your API key and CRN where indicated, run **once**, then re-comment. Your credentials get saved to `~/.qiskit/qiskit-ibm.json` and every future Python session on this machine will pick them up automatically.

### Common gotchas

- **"Unable to find a default account"** — you didn't run the save cell, or ran it in a different Python environment.
- **`.backends()` returns `[]`** — you skipped step 2 (creating the instance).
- **"Could not verify IBM Cloud credentials"** — wrong API key, or the CRN is missing the trailing `::`.
- **Forgot to download the key in time** — go back to the API key page, delete the entry, create a new one.

In [ ]:
# === IBM Account Setup  (run ONCE, then re-comment) ===
# Uncomment the lines below, paste your API key and CRN, and run this cell ONCE.
# After that, re-comment these lines — your credentials are saved at
# ~/.qiskit/qiskit-ibm.json and all future sessions pick them up automatically.

# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel='ibm_cloud',
#     token='YOUR_API_KEY_HERE',          # the apikey field from apikey.json
#     instance='YOUR_INSTANCE_CRN_HERE',  # ends with ::
#     overwrite=True,
#     set_as_default=True,
# )

In [ ]:
# === Connect to IBM Quantum and pick the least-busy QPU ===
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

service = QiskitRuntimeService()    # picks up saved credentials automatically
backend = service.least_busy(operational=True, simulator=False)

# Holds your hardware job IDs so you can recover results if the kernel restarts.
job_ids = {}
print(f"Connected! Using backend: {backend.name}  ({backend.num_qubits} qubits)")

In [ ]:
# === Sanity check: confirm we can see backends and the chosen one is real ===
print("Available backends:", [b.name for b in service.backends()])
print(f"Selected: {backend.name}")
print(f"  status:  {backend.status().status_msg}")
print(f"  queue:   {backend.status().pending_jobs} jobs ahead of yours")
print(f"  qubits:  {backend.num_qubits}")

---
## Part C — Run Grover's on Real Hardware

For each n (2, 3, and 4), we'll transpile the circuit for the chosen backend, submit, and compare hardware success probability against the perfect simulator.

In [ ]:
# === Transpile all three Grover circuits for the real backend ===
# This step rewrites your circuits to use only the gates the chip natively supports
# (CZ, RZ, SX, X for Heron-class devices), maps your logical qubits to specific
# physical qubits on the chip, and inserts SWAP gates if the chosen qubits aren't
# directly connected.
pm = generate_preset_pass_manager(target=backend.target, optimization_level=3)
grover_n2_t = pm.run(grover_n2)
grover_n3_t = pm.run(grover_n3)
grover_n4_t = pm.run(grover_n4)

print(f"n=2:  depth before = {grover_n2.depth():>3d}   →   after transpile = {grover_n2_t.depth():>4d}")
print(f"n=3:  depth before = {grover_n3.depth():>3d}   →   after transpile = {grover_n3_t.depth():>4d}")
print(f"n=4:  depth before = {grover_n4.depth():>3d}   →   after transpile = {grover_n4_t.depth():>4d}")

print("""
Notice the depth explosion. n=2 barely changes (CZ is native and the qubits are adjacent),
but n=3 and n=4 grow because:
  - The multi-controlled Z (MCMTGate(Z, n-1, 1)) has to be decomposed into many CZs
  - The chosen physical qubits may need SWAP gates to become adjacent
  - For n=4 the circuit is so deep that hardware success will likely be near 1/16 ≈ 0.06
    (i.e. essentially random). That is not a bug — it is the headline result of this project.
This is what kills hardware success: every extra gate is more time for decoherence.
""")

# Visualize what the transpiled n=3 circuit actually looks like (eye-opening!)
grover_n3_t.draw('mpl', fold=-1, idle_wires=False)

In [ ]:
# === Submit Grover circuits to hardware ===
sampler = Sampler(mode=backend)
job = sampler.run([grover_n2_t, grover_n3_t, grover_n4_t], shots=SHOTS)
print(f"Submitted! Job ID: {job.job_id()}")    # <-- copy this immediately
job_ids['part_c_main'] = job.job_id()

# The next line BLOCKS until the job finishes — could be seconds or hours
# depending on queue load. The notebook will appear "stuck" until then.
# That is normal. If you need to come back later, recover via the next
# section using the job ID printed above.
result = job.result()
hw_counts = {
    'n=2': result[0].data.c.get_counts(),
    'n=3': result[1].data.c.get_counts(),
    'n=4': result[2].data.c.get_counts(),
}

for name, counts in hw_counts.items():
    print(f"{name}  counts: {dict(sorted(counts.items(), key=lambda kv: -kv[1])[:5])}  ...")

### If you closed your laptop and lost the result variable

Hardware jobs sit in a queue and may not return for hours. If you submit, close the notebook,
and come back later, the `result` variable is gone from kernel memory — but the job is still
running on IBM's servers. Use the job ID you saved in `job_ids` to recover the results without
re-submitting (and without spending more QPU time):


In [ ]:
# === Recover a job by ID (use this if your kernel restarted) ===
# Uncomment, paste your job ID, and run:
#
# saved_job_id = "PASTE_YOUR_JOB_ID_HERE"
# job  = service.job(saved_job_id)
# print(f"Status: {job.status()}")        # 'DONE' = ready; 'RUNNING'/'QUEUED' = wait
# if job.status() == 'DONE':
#     result = job.result()
#     hw_counts = {
#         'n=2': result[0].data.c.get_counts(),
#         'n=3': result[1].data.c.get_counts(),
#         'n=4': result[2].data.c.get_counts(),
#     }
#     print("Recovered. You can now re-run the success table and plot cells.")

In [ ]:
# === Success probability table ===
success_hw = {
    'n=2': hw_counts['n=2'].get('11',   0) / SHOTS,
    'n=3': hw_counts['n=3'].get('101',  0) / SHOTS,
    'n=4': hw_counts['n=4'].get('1010', 0) / SHOTS,
}

print(f"{'n':>4s}  {'P sim':>8s}  {'P hw':>8s}  {'1/N':>8s}  {'depth':>8s}")
print("-" * 46)
for n, qc_t in [(2, grover_n2_t), (3, grover_n3_t), (4, grover_n4_t)]:
    N = 2**n
    print(f"{n:>4d}  {success_sim[f'n={n}']:>8.3f}  {success_hw[f'n={n}']:>8.3f}  "
          f"{1/N:>8.3f}  {qc_t.depth():>8d}")

In [ ]:
# === Headline plot 1: success probability vs. n ===
n_vals = np.array([2, 3, 4])
p_sim  = np.array([success_sim[f'n={n}'] for n in n_vals])
p_hw   = np.array([success_hw[f'n={n}']  for n in n_vals])
p_rand = 1 / 2.0**n_vals
depths = np.array([grover_n2_t.depth(), grover_n3_t.depth(), grover_n4_t.depth()])

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(n_vals, p_sim,  'o-', label='Simulator (ideal)',     markersize=10)
ax.plot(n_vals, p_hw,   's-', label=f'Hardware ({backend.name})', markersize=10)
ax.plot(n_vals, p_rand, 'k--', alpha=0.5, label='Random guessing (1/N)')
ax.set_xlabel('Number of qubits  n')
ax.set_ylabel('Success probability  P(target)')
ax.set_title("Grover's success — simulator vs. real hardware")
ax.set_xticks(n_vals)
ax.set_ylim(-0.05, 1.05)
ax.grid(True, alpha=0.3)
# Annotate each hardware point with the transpiled depth
for n, p, d in zip(n_vals, p_hw, depths):
    ax.annotate(f'depth={d}', xy=(n, p), xytext=(0, -16),
                textcoords='offset points', ha='center', fontsize=9, color='gray')
ax.legend()
plt.tight_layout()
plt.show()

---
## Part D — Iteration Sweep at n=3

For a fixed $n$, the success probability follows a sin² overshoot curve as you vary the number of Grover iterations $t$:

$$P(\text{target}, t) \;=\; \sin^2\bigl((2t+1)\theta\bigr), \qquad \theta = \arcsin(1/\sqrt{N})$$

You verified this on the simulator in HW 4.6 Part C2. Now we'll see what happens to that curve on real hardware. We'll sweep $t = 0, 1, \ldots, 6$ at $n=3$ (optimal is $t \approx 2$).

In [ ]:
# === Build n=3 Grover circuits for t = 0..6 and run on simulator ===
t_vals = list(range(7))
sweep_circuits = [build_grover("101", 3, t=t) for t in t_vals]

# Theoretical curve
theta    = math.asin(1 / math.sqrt(8))
P_theory = np.sin((2 * np.array(t_vals) + 1) * theta) ** 2

sweep_sim = []
for t, qc in zip(t_vals, sweep_circuits):
    counts = simulator.run(transpile(qc, simulator), shots=SHOTS).result().get_counts()
    p = counts.get('101', 0) / SHOTS
    sweep_sim.append(p)
    print(f"t = {t}:   sim P(101) = {p:.3f}    (theory = {P_theory[t]:.3f})")

In [ ]:
# === Submit the same sweep to hardware, then plot all three curves ===
sweep_circuits_t = [pm.run(qc) for qc in sweep_circuits]
sampler = Sampler(mode=backend)
job = sampler.run(sweep_circuits_t, shots=SHOTS)
print(f"Submitted sweep! Job ID: {job.job_id()}")
job_ids['part_d_sweep'] = job.job_id()
result = job.result()
sweep_hw = [result[i].data.c.get_counts().get('101', 0) / SHOTS
            for i in range(len(t_vals))]

# === Headline plot 2: iteration sweep ===
plt.figure(figsize=(8, 5))
plt.plot(t_vals, P_theory, 'k--', alpha=0.5, label=r'Theory: $\sin^2((2t+1)\theta)$')
plt.plot(t_vals, sweep_sim, 'o-', label='Simulator (ideal)', markersize=8)
plt.plot(t_vals, sweep_hw,  's-', label=f'Hardware ({backend.name})', markersize=8)
plt.axvline(optimal_iterations(3), color='red', linestyle=':', alpha=0.5,
            label=f't_optimal = {optimal_iterations(3)}')
plt.xlabel('Number of Grover iterations  t')
plt.ylabel('Success probability  P(101)')
plt.title("n=3 iteration sweep — does the overshoot survive on hardware?")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Part E — Backend Properties and Analysis

In [ ]:
# === Pull the backend's calibration data for THE QUBITS YOUR CIRCUIT ACTUALLY USED ===
# Important: the transpiler picked specific physical qubits to run your circuit on.
# Those particular qubits' T1, T2, and gate errors are what your noise comes from —
# not the median across all 156 qubits. We extract them per-circuit.

target = backend.target

def circuit_specific_props(transpiled_circuit, label):
    """Return T1/T2/readout/2Q-error stats for the qubits this circuit used."""
    # Which physical qubits did the transpiler actually map onto?
    used_phys = sorted({transpiled_circuit.find_bit(q).index
                        for inst in transpiled_circuit.data for q in inst.qubits})

    t1s = [target.qubit_properties[q].t1 * 1e6 for q in used_phys
           if target.qubit_properties[q] and target.qubit_properties[q].t1]
    t2s = [target.qubit_properties[q].t2 * 1e6 for q in used_phys
           if target.qubit_properties[q] and target.qubit_properties[q].t2]
    readouts = [target['measure'][(q,)].error for q in used_phys
                if (q,) in target['measure'] and target['measure'][(q,)].error is not None]

    two_q_name = next((n for n in ('cz', 'ecr', 'cx') if n in target.operation_names), None)
    # 2Q errors only on the pairs that this circuit actually exercised
    used_pairs = {tuple(sorted((transpiled_circuit.find_bit(q1).index,
                                transpiled_circuit.find_bit(q2).index)))
                  for inst in transpiled_circuit.data
                  if len(inst.qubits) == 2
                  for (q1, q2) in [inst.qubits]}
    two_q_errs = []
    for (a, b) in used_pairs:
        for pair in ((a, b), (b, a)):
            if pair in target[two_q_name] and target[two_q_name][pair].error is not None:
                two_q_errs.append(target[two_q_name][pair].error)
                break

    print(f"\n--- {label}  (depth {transpiled_circuit.depth()}, qubits {used_phys}) ---")
    print(f"  T1 (median over used qubits):           {np.median(t1s):6.1f} µs")
    print(f"  T2 (median over used qubits):           {np.median(t2s):6.1f} µs")
    print(f"  readout error (median):                 {np.median(readouts):.2e}")
    print(f"  {two_q_name.upper()} gate error (median over used pairs): {np.median(two_q_errs):.2e}")

    return {
        'depth': transpiled_circuit.depth(),
        'qubits': used_phys,
        'T1_us': float(np.median(t1s)),
        'T2_us': float(np.median(t2s)),
        'readout': float(np.median(readouts)),
        f'{two_q_name}_err': float(np.median(two_q_errs)) if two_q_errs else None,
    }

print(f"Backend: {backend.name}  (Heron-class, {backend.num_qubits} total qubits)")
specs_n2 = circuit_specific_props(grover_n2_t, 'n=2 circuit')
specs_n3 = circuit_specific_props(grover_n3_t, 'n=3 circuit')
specs_n4 = circuit_specific_props(grover_n4_t, 'n=4 circuit')

---
## Part F — Memo

You've been asked to write a one-page memo to a (hypothetical) lab director who is deciding whether to run a larger Grover's search (n=6) on this backend next month. They understand quantum computing but haven't seen your data.

Write the memo (aim for **200–250 words**). It must:

- Open with a **one-sentence bottom line** (yes/no recommendation, **not** "it depends")
- Summarize what you found at n=2, 3, and 4 — P_hw, depth, and the key failure mode — in **no more than three sentences**
- Make a **back-of-envelope prediction for n=6**: estimate the transpiled depth (depth scales roughly as $O(n \cdot 4^n)$ for the MCZ decomposition), compute the error budget, and state whether P_hw would likely be above or below random
- Name the **single hardware spec** that would need to improve most to make n=6 viable, and by roughly what factor
- Close with a sentence about what the **iteration-sweep result** tells you about whether the problem is fixable by recalibration alone


**Your memo (double-click to edit, target 200–250 words):**

---

**To:** Lab Director  
**From:** [Your Name]  
**Subject:** Recommendation on running n=6 Grover's search on this backend next month

**Bottom line:**

**What we found at n=2, 3, and 4:**

**Prediction for n=6:**

**Hardware spec that would need to improve most:**

**Iteration-sweep takeaway:**

---


---
## Part G — Experimental Design

IBM's Open Plan gives you 10 QPU minutes per month, but suppose a lab account gives you 50. You want to use this project's data to answer a question that your current three data points (n=2, 3, 4) cannot answer on their own:

> *At what circuit depth does this specific backend cross the "quantum uselessness threshold" — the point where $P_{hw} \le 1/N$?*

Your n=3 and n=4 results bracket the threshold somewhere, but you don't know exactly where. Design an experiment to pin it down, subject to these constraints:

- You **cannot** increase n beyond 4 (too many QPU minutes per job)
- You **can** vary $t$ (number of iterations) at fixed n=3, which changes depth without changing n
- Each hardware job costs roughly 1 QPU minute per circuit submitted

Your design should specify:

- **Which circuits** you would submit (what values of $t$, and why those specifically)
- **How many QPU minutes** your plan uses (out of 50 available)
- **What data** you would collect and **what plot** you would make
- **How you would define "crossed the threshold"** from noisy data — a single shot where $P_{hw} < 1/N$ doesn't count, so what statistical criterion would you use?

There is no single right answer, but a good answer will show you understand that depth and $t$ are related, that noise makes the threshold fuzzy, and that QPU minutes are finite.


**Your design (double-click to edit):**

**Circuits to submit (t values + reasoning):**

**Total QPU minutes used:**

**Data to collect + plot you would make:**

**Threshold-crossing criterion:**


---
## Submission Checklist

Before submitting, verify:

- [ ] `build_oracle` and `build_diffusion` working (n=2, n=3, and n=4 all pass simulator sanity check)
- [ ] All three Grover circuits run on the simulator (P > 0.9 for n=2 and n=3, > 0.8 for n=4)
- [ ] All three Grover circuits run on real IBM hardware
- [ ] Iteration sweep run on both simulator and real hardware (n=3, t = 0..6)
- [ ] **Headline plot 1** (success vs. n) included
- [ ] **Headline plot 2** (iteration sweep, all three curves) included
- [ ] Backend properties cell run and numbers recorded above
- [ ] **Memo** (Part F, 200–250 words): bottom line, n=2/3/4 summary, n=6 prediction, hardware spec, iteration-sweep takeaway
- [ ] **Experimental design** (Part G): circuits to submit, QPU minutes used, data + plot, threshold-crossing criterion
- [ ] **IBM Job IDs recorded below** (proof of hardware execution)

In [ ]:
# Record your IBM job IDs here (copy from the Sampler print statements above)
for label, jid in job_ids.items():
    print(f"  {label}: {jid}")

---

**End of Final Project** — good luck, and email me before May 8 if you hit a wall.